# Employee Attrition Analysis

## Project Overview
Analyze employee data to identify attrition patterns across tenure, department, compensation, manager, and engagement variables. This is a descriptive analytics project.

## Learning Objectives
- Calculate attrition rates across multiple HR dimensions
- Identify high-risk employee segments for attrition
- Analyze the relationship between engagement, compensation, and turnover
- Detect temporal patterns in employee departures

## Problem Statement
HR leadership is concerned about rising turnover. They need data to understand which departments, tenure bands, and employee profiles are most at risk, and what factors correlate with attrition.

## Why This Project Matters
Employee attrition is expensive — replacement costs range from 50-200% of annual salary. Understanding attrition drivers enables targeted retention strategies.

## Dataset Overview
Synthetic HR dataset: ~3,000 employees with demographics, compensation, tenure, engagement scores, and attrition status.

## Dataset Source and License Notes
- Synthetic data inspired by IBM HR Analytics dataset structure
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 3000
departments = np.random.choice(['Engineering', 'Sales', 'Marketing', 'Operations', 'Finance', 'HR', 'Support'],
                                 n, p=[0.22, 0.20, 0.12, 0.15, 0.10, 0.08, 0.13])
levels = np.random.choice(['Junior', 'Mid', 'Senior', 'Lead', 'Manager'], n, p=[0.25, 0.30, 0.25, 0.12, 0.08])
tenure_months = np.clip(np.random.exponential(30, n).astype(int), 1, 180)
age = np.clip(np.random.normal(35, 8, n).astype(int), 22, 62)
gender = np.random.choice(['Male', 'Female', 'Non-Binary'], n, p=[0.52, 0.45, 0.03])
engagement = np.clip(np.random.normal(3.5, 0.8, n), 1.0, 5.0).round(1)
salary = np.clip(np.random.lognormal(11.0, 0.35, n), 35000, 250000).round(0)
perf_rating = np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.15, 0.40, 0.30, 0.10])
manager_id = np.random.choice([f'MGR{i:03d}' for i in range(50)], n)
remote = np.random.choice(['Remote', 'Hybrid', 'On-Site'], n, p=[0.30, 0.40, 0.30])

# Attrition logic
attrition_prob = 0.08 +     0.12 * (engagement < 2.5) +     0.08 * (tenure_months < 12) +     0.06 * (perf_rating <= 2) +     0.05 * (salary < 50000) +     np.random.normal(0, 0.02, n)
attrition_prob = np.clip(attrition_prob, 0.02, 0.60)
attrited = (np.random.random(n) < attrition_prob).astype(int)

df = pd.DataFrame({
    'EmployeeID': [f'E{i:05d}' for i in range(n)],
    'Department': departments, 'Level': levels,
    'TenureMonths': tenure_months, 'Age': age, 'Gender': gender,
    'EngagementScore': engagement, 'Salary': salary,
    'PerformanceRating': perf_rating, 'ManagerID': manager_id,
    'WorkMode': remote, 'Attrited': attrited,
})
df['TenureBand'] = pd.cut(df['TenureMonths'], bins=[0, 6, 12, 24, 48, 999],
                            labels=['0-6mo', '6-12mo', '1-2yr', '2-4yr', '4yr+'])
df['SalaryBand'] = pd.qcut(df['Salary'], q=4, labels=['Q1-Low', 'Q2', 'Q3', 'Q4-High'])

print(f'Dataset shape: {df.shape}')
print(f'Overall attrition rate: {df["Attrited"].mean():.1%}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Employees: {df["EmployeeID"].nunique()}')
print(f'\nDepartment distribution:\n{df["Department"].value_counts()}')
print(f'\nAttrition counts:\n{df["Attrited"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df.groupby('Department')['Attrited'].mean().sort_values().plot.barh(ax=axes[0,0], edgecolor='black', color='coral')
axes[0,0].set_title('Attrition Rate by Department')

df.groupby('TenureBand')['Attrited'].mean().plot.bar(ax=axes[0,1], edgecolor='black', color='steelblue')
axes[0,1].set_title('Attrition Rate by Tenure Band')
axes[0,1].tick_params(axis='x', rotation=0)

df.groupby('Level')['Attrited'].mean().reindex(['Junior','Mid','Senior','Lead','Manager']).plot.bar(
    ax=axes[1,0], edgecolor='black', color='green')
axes[1,0].set_title('Attrition Rate by Level')
axes[1,0].tick_params(axis='x', rotation=0)

for label, grp in df.groupby('Attrited'):
    axes[1,1].hist(grp['EngagementScore'], bins=20, alpha=0.5, edgecolor='black',
                   label='Stayed' if label == 0 else 'Left')
axes[1,1].set_title('Engagement Score Distribution')
axes[1,1].legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Attrition by Compensation

In [ ]:
sal_attrition = df.groupby('SalaryBand').agg(
    employees=('EmployeeID', 'count'),
    attrition_rate=('Attrited', 'mean'),
    avg_salary=('Salary', 'mean'),
    avg_engagement=('EngagementScore', 'mean'),
).round(3)
print('Attrition by Salary Band:')
print(sal_attrition)

fig, ax = plt.subplots(figsize=(8, 5))
sal_attrition['attrition_rate'].plot.bar(ax=ax, edgecolor='black', color='coral')
ax.set_title('Attrition Rate by Salary Quartile')
ax.set_ylabel('Attrition Rate')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'salary_attrition.png'), dpi=100, bbox_inches='tight')
plt.show()

## Engagement vs Performance Interaction

In [ ]:
df['EngBucket'] = pd.cut(df['EngagementScore'], bins=[0, 2.5, 3.5, 4.5, 5.1],
                          labels=['Low', 'Medium', 'High', 'Very High'])
cross = df.groupby(['EngBucket', 'PerformanceRating'])['Attrited'].mean().unstack().round(3)
print('Attrition Rate — Engagement × Performance:')
print(cross)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(cross, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax)
ax.set_title('Attrition Rate: Engagement × Performance Rating')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'engagement_performance.png'), dpi=100, bbox_inches='tight')
plt.show()

## Manager Impact on Attrition

In [ ]:
mgr_stats = df.groupby('ManagerID').agg(
    reports=('EmployeeID', 'count'),
    attrition_rate=('Attrited', 'mean'),
    avg_engagement=('EngagementScore', 'mean'),
).round(3)
mgr_stats = mgr_stats[mgr_stats['reports'] >= 10]  # min sample
mgr_worst = mgr_stats.sort_values('attrition_rate', ascending=False).head(10)
print(f'Managers with 10+ reports: {len(mgr_stats)}')
print('\nTop 10 Highest Attrition Managers:')
print(mgr_worst)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(mgr_stats['avg_engagement'], mgr_stats['attrition_rate'],
          s=mgr_stats['reports'] * 3, alpha=0.5, edgecolor='black', color='steelblue')
ax.set_xlabel('Avg Team Engagement')
ax.set_ylabel('Attrition Rate')
ax.set_title('Manager: Team Engagement vs Attrition (bubble = team size)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'manager_impact.png'), dpi=100, bbox_inches='tight')
plt.show()

## Work Mode Analysis

In [ ]:
wm = df.groupby('WorkMode').agg(
    employees=('EmployeeID', 'count'),
    attrition_rate=('Attrited', 'mean'),
    avg_engagement=('EngagementScore', 'mean'),
).round(3)
print('Attrition by Work Mode:')
print(wm)

fig, ax = plt.subplots(figsize=(8, 5))
wm['attrition_rate'].plot.bar(ax=ax, edgecolor='black', color='teal')
ax.set_title('Attrition Rate by Work Mode')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'work_mode.png'), dpi=100, bbox_inches='tight')
plt.show()

## Attrited vs Stayed Profile Comparison

In [ ]:
profile_cols = ['Age', 'TenureMonths', 'Salary', 'EngagementScore', 'PerformanceRating']
profile = df.groupby('Attrited')[profile_cols].mean()
profile.index = ['Stayed', 'Left']
print('Average Profile — Stayed vs Left:')
print(profile.T.round(2))

## Interpretation and Business Insight
- **Low engagement** is the strongest attrition predictor — employees scoring <2.5 leave at dramatically higher rates
- **Short tenure** (<12 months) shows elevated attrition — onboarding and early experience are critical
- **Compensation** matters: lowest salary quartile has noticeably higher turnover
- **Manager quality** correlates with team attrition — some managers have 2-3x the average attrition rate
- **Performance rating** interacts with engagement: low-performing, disengaged employees are highest risk, but high performers with low engagement also leave (flight risk)
- **Work mode** differences are modest but Remote shows slightly different patterns

## Limitations
- Synthetic data with pre-programmed attrition logic
- No exit interview or reason-for-leaving data
- No time-series component (snapshot analysis only)
- No benefits, promotion history, or career progression data
- Manager assignments are random — real manager-report relationships are more structured

## How to Improve This Project
- Use real HRIS data with exit surveys
- Add survival analysis for tenure-based attrition modeling
- Include promotion and compensation change history
- Build predictive attrition models (logistic regression, gradient boosting)
- Add cost-of-attrition quantification by role level

## Production Considerations
- Monthly attrition risk dashboards for HR business partners
- Automated flight risk alerts for high-value employees
- Manager scorecards including team engagement and attrition metrics
- Integration with engagement survey platforms for real-time signals

## Common Mistakes
- Using overall attrition rate without segmenting by tenure, role, or department
- Confusing voluntary and involuntary attrition (very different drivers)
- Not controlling for headcount when comparing departments
- Ignoring survivorship bias in engagement scores (disengaged people already left)

## Mini Challenge / Exercises
1. Calculate the regrettable attrition rate (high performers who left). What % of all attrition is regrettable?
2. Which department × level combination has the highest attrition rate?
3. If you improved all Low engagement scores to Medium, estimate the reduction in overall attrition.
4. Calculate the estimated replacement cost assuming 1.5× annual salary per departure.

## Final Summary / Key Takeaways
- Employee attrition analysis reveals actionable patterns in who leaves and why
- Engagement, tenure, and compensation are the three strongest levers
- Manager quality is an underappreciated attrition driver
- Segmented analysis (department × level × tenure) is far more useful than aggregate rates
- Proactive retention programs should target identified high-risk segments